# Offroad Semantic Segmentation — Colab Training

> **Before running:** `Runtime → Change runtime type → T4 GPU`

### Your Google Drive should have these two zip files:
```
My Drive/
  Offroad_Segmentation_Training_Dataset.zip   ← train + val images
  Offroad_Segmentation_testImages.zip          ← test images
```

### Run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU found!\n"
        "Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
# ── Cell 3: Find zip files on Drive ──────────────────────────────────
import os
from pathlib import Path

# ── If your zips are in a subfolder, set the folder path here ─────────
DRIVE_SEARCH_ROOT = "/content/drive/MyDrive"
# ─────────────────────────────────────────────────────────────────────

TRAIN_ZIP_NAME = "Offroad_Segmentation_Training_Dataset.zip"
TEST_ZIP_NAME  = "Offroad_Segmentation_testImages.zip"

def find_file(root, name):
    """Search recursively under root for a file with the given name."""
    for dirpath, dirnames, filenames in os.walk(root):
        # Skip hidden and system dirs
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        if name in filenames:
            return os.path.join(dirpath, name)
    return None

print(f"Searching for zips under {DRIVE_SEARCH_ROOT} ...")
TRAIN_ZIP = find_file(DRIVE_SEARCH_ROOT, TRAIN_ZIP_NAME)
TEST_ZIP  = find_file(DRIVE_SEARCH_ROOT, TEST_ZIP_NAME)

if not TRAIN_ZIP:
    raise FileNotFoundError(
        f"{TRAIN_ZIP_NAME} not found under {DRIVE_SEARCH_ROOT}.\n"
        "Make sure you uploaded it to Google Drive."
    )

print(f"Found training zip : {TRAIN_ZIP}")
print(f"Found test zip     : {TEST_ZIP if TEST_ZIP else 'NOT FOUND (optional)'}")

In [ ]:
# ── Cell 4: Extract zips to fast local Colab storage ─────────────────
# Copying to /content (local SSD) is ~3x faster than reading from Drive
# during training. This cell takes ~2-3 minutes.

import zipfile, os

EXTRACT_ROOT = "/content/dataset"

def extract_zip(zip_path, dest):
    print(f"Extracting {os.path.basename(zip_path)} → {dest}")
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest)
    print(f"  Done.")

def find_subfolder(root, target):
    """Walk root and return the first path that contains target as a direct child."""
    for dirpath, dirnames, _ in os.walk(root):
        if target in dirnames:
            return os.path.join(dirpath, target)
    return None

# ── Training dataset ──────────────────────────────────────────────────
TRAIN_EXTRACT = os.path.join(EXTRACT_ROOT, "training")
if not os.path.exists(TRAIN_EXTRACT):
    extract_zip(TRAIN_ZIP, TRAIN_EXTRACT)
else:
    print(f"Training data already extracted at {TRAIN_EXTRACT}")

# Auto-detect DATASET_BASE (handles double-nesting like Foo/Foo/train/)
train_color = find_subfolder(TRAIN_EXTRACT, "Color_Images")
if train_color is None:
    # try one level up
    for d in sorted(Path(TRAIN_EXTRACT).rglob("Color_Images")):
        train_color = str(d)
        break
if train_color is None:
    raise RuntimeError("Could not find Color_Images inside the training zip. Check the zip structure.")

# DATASET_BASE is the parent of the folder containing Color_Images
# e.g. .../training/Offroad_.../Offroad_.../train/Color_Images  →  .../train/  →  parent
DATASET_BASE = str(Path(train_color).parent.parent)  # up from Color_Images, up from train/

# ── Test images (optional) ────────────────────────────────────────────
TEST_EXTRACT = os.path.join(EXTRACT_ROOT, "test")
TEST_IMAGES_PATH = None
if TEST_ZIP and not os.path.exists(TEST_EXTRACT):
    extract_zip(TEST_ZIP, TEST_EXTRACT)
elif os.path.exists(TEST_EXTRACT):
    print(f"Test data already extracted at {TEST_EXTRACT}")

if os.path.exists(TEST_EXTRACT):
    for d in sorted(Path(TEST_EXTRACT).rglob("Color_Images")):
        TEST_IMAGES_PATH = str(d)
        break

# ── Verify counts ─────────────────────────────────────────────────────
print(f"\nDATASET_BASE = {DATASET_BASE}")
for split in ["train", "val"]:
    for sub in ["Color_Images", "Segmentation"]:
        p = os.path.join(DATASET_BASE, split, sub)
        n = len([f for f in os.listdir(p) if f.endswith('.png')]) if os.path.exists(p) else -1
        status = f"{n} images" if n > 0 else "NOT FOUND"
        print(f"  {split}/{sub}: {status}")

if TEST_IMAGES_PATH:
    n = len([f for f in os.listdir(TEST_IMAGES_PATH) if f.endswith('.png')])
    print(f"  testImages/Color_Images: {n} images")

In [ ]:
# ── Cell 5: Clone project from GitHub ────────────────────────────────
import os

REPO_URL = "https://github.com/kritikamandale/HackDaysNagpur.git"
REPO_DIR = "/content/HackDaysNagpur"
PROJECT  = "/content/HackDaysNagpur/offroad_seg"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(PROJECT)
print(f"\nWorking directory: {os.getcwd()}")
!ls

In [ ]:
# ── Cell 6: Install dependencies ─────────────────────────────────────
# Colab already has: torch, torchvision, numpy, Pillow, matplotlib, tqdm
!pip install -q \
    "segmentation-models-pytorch>=0.3.3" \
    "albumentations==1.3.1" \
    "timm==0.9.2" \
    opencv-python-headless \
    PyYAML

# Uncomment to enable W&B logging:
# !pip install -q wandb

print("\nVerifying installs...")
import torch, segmentation_models_pytorch as smp, albumentations as A, cv2
print(f"  torch {torch.__version__} | smp {smp.__version__} | albumentations {A.__version__} | cv2 {cv2.__version__}")

In [ ]:
# ── Cell 7: Configure paths & hyperparameters ─────────────────────────
import yaml, os

cfg_path = "configs/config.yaml"
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Data paths (set by Cell 4)
cfg["data"]["train_rgb"]   = f"{DATASET_BASE}/train/Color_Images"
cfg["data"]["train_masks"] = f"{DATASET_BASE}/train/Segmentation"
cfg["data"]["val_rgb"]     = f"{DATASET_BASE}/val/Color_Images"
cfg["data"]["val_masks"]   = f"{DATASET_BASE}/val/Segmentation"
cfg["data"]["test_images"] = TEST_IMAGES_PATH or f"{DATASET_BASE}/testImages"

# Training settings — tuned for Colab T4 GPU
cfg["device"]              = "cuda"
cfg["train"]["epochs"]     = 30      # ~45-60 min on T4
cfg["train"]["batch_size"] = 8       # T4 handles 8 easily at 256x256
cfg["train"]["image_size"] = [256, 256]
cfg["train"]["save_dir"]   = "runs/"
cfg["scheduler"]["T_max"]  = cfg["train"]["epochs"]

os.makedirs("runs", exist_ok=True)

with open(cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print("Config updated:")
print(f"  model      : {cfg['model']['architecture']} + {cfg['model']['encoder']}")
print(f"  epochs     : {cfg['train']['epochs']}")
print(f"  batch_size : {cfg['train']['batch_size']}")
print(f"  image_size : {cfg['train']['image_size']}")
print(f"  device     : {cfg['device']}")
print(f"  train_rgb  : {cfg['data']['train_rgb']}")
print(f"  val_rgb    : {cfg['data']['val_rgb']}")

In [ ]:
# ── Cell 8: TRAIN ─────────────────────────────────────────────────────
# Expected: ~1-2 min/epoch on T4 → 30 epochs ≈ 45-60 min total
# Best checkpoint auto-saves to runs/best_model.pth

!python train.py

# With W&B logging (install wandb in Cell 6 first, then run: !wandb login):
# !python train.py --wandb

In [ ]:
# ── Cell 9: Save results to Google Drive ──────────────────────────────
import shutil, os

SAVE_DIR = "/content/drive/MyDrive/offroad_seg_results"
os.makedirs(SAVE_DIR, exist_ok=True)

for src, dst in [
    ("runs/best_model.pth",      "best_model.pth"),
    ("runs/training_curves.png", "training_curves.png"),
]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SAVE_DIR, dst))
        mb = os.path.getsize(src) / 1e6
        print(f"  Saved {dst}  ({mb:.1f} MB)")
    else:
        print(f"  [SKIP] {src} not found")

print(f"\nAll saved to: {SAVE_DIR}")
print("Download best_model.pth and place it at offroad_seg/runs/best_model.pth to use the app locally.")

In [ ]:
# ── Cell 10 (optional): Final validation metrics ──────────────────────
import torch, sys
sys.path.insert(0, '.')

from src.utils      import load_config, print_iou_table
from src.model      import load_model_for_inference
from src.dataset    import SegmentationDataset
from src.transforms import get_val_transforms
from src.losses     import build_loss
from src.metrics    import IoUMeter
from torch.utils.data import DataLoader
from train import validate

cfg    = load_config("configs/config.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = load_model_for_inference("runs/best_model.pth", cfg, str(device))

val_ds     = SegmentationDataset(cfg["data"]["val_rgb"], cfg["data"]["val_masks"],
                                  get_val_transforms(cfg["train"]["image_size"]))
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
loss_fn    = build_loss(cfg, device)
val_meter  = IoUMeter(cfg["classes"]["num_classes"])

val_loss, val_iou, per_class_iou = validate(model, val_loader, loss_fn, val_meter, device)
print(f"Val loss : {val_loss:.4f}")
print(f"Val mIoU : {val_iou:.4f}")
print_iou_table(per_class_iou, val_iou)